<a href="https://colab.research.google.com/github/KeshavFTW099/ArtificalAdvanced_NeuralNetworks/blob/main/RESNet_CIFAR10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input

from tensorflow.keras.layers import Input, Resizing, Lambda, Dense, Flatten, GlobalAveragePooling2D
from tensorflow.keras.layers import BatchNormalization, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import plot_model

In [3]:
import tensorflow as tf
from tensorflow.keras.datasets import cifar10

# Load the CIFAR-10 dataset
(x_train, y_train), (x_test, y_test) = cifar10.load_data()

x_train = x_train.astype('float32')
x_test = x_test.astype('float32')

# Images will be resized and preprocessed within the model to avoid OOM

print(f"Shape of x_train: {x_train.shape}")
print(f"Shape of y_train: {y_train.shape}")
print(f"Shape of x_test: {x_test.shape}")
print(f"Shape of y_test: {y_test.shape}")

Shape of x_train: (50000, 32, 32, 3)
Shape of y_train: (50000, 1)
Shape of x_test: (10000, 32, 32, 3)
Shape of y_test: (10000, 1)


Train/Validation Split + One Hot Encoding + Augmentation

In [4]:
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

num_classes = 10

x_train, x_val, y_train, y_val = train_test_split(
    x_train,
    y_train,
    test_size=0.1,
    random_state=42
)

y_train = to_categorical(y_train, num_classes)
y_val = to_categorical(y_val, num_classes)
y_test = to_categorical(y_test, num_classes)


train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1
)

val_datagen = ImageDataGenerator()

train_generator = train_datagen.flow(
    x_train,
    y_train,
    batch_size=32,
    shuffle=True
)

val_generator = val_datagen.flow(
    x_val,
    y_val,
    batch_size=32,
    shuffle=False
)

test_datagen = ImageDataGenerator()

test_generator = test_datagen.flow(
    x_test,
    y_test,
    batch_size=32,
    shuffle=False
)

ResNet50 + Fine Tuning + Dense Architecture

In [5]:
# Define the input layer for the model, matching the original CIFAR-10 image size (32x32x3)
input_tensor = Input(shape=(32, 32, 3))

# Resize the images to 224x224 dynamically within the model
x = Resizing(224, 224)(input_tensor)

# Apply ResNet50's specific preprocessing function
x = Lambda(preprocess_input)(x)

# Load the ResNet50 model with the preprocessed input
resnet_base = ResNet50(
    input_tensor=x, # Connect the preprocessed input here
    include_top=False,
    weights='imagenet'
)

for layer in resnet_base.layers:
    layer.trainable = False

num_layers_to_unfreeze = 10

for layer in resnet_base.layers[-num_layers_to_unfreeze:]:
    layer.trainable = True

x = resnet_base.output

x = GlobalAveragePooling2D()(x)

x = Dense(1024, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(512, activation='relu')(x)
x = BatchNormalization()(x)
x = Dropout(0.5)(x)

x = Dense(256, activation='relu')(x)
x = Dropout(0.2)(x)

output = Dense(10, activation='softmax')(x)

model = Model(
    inputs=input_tensor, # The overall model's input is the original 32x32x3
    outputs=output
)

94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


Compile + Reduce LR + Early Stopping + Training

In [7]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.2,
    patience=3,
    min_lr=0.000001
)


early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)


history = model.fit(
    train_generator,
    epochs=10,
    validation_data=val_generator,
    callbacks=[reduce_lr, early_stopping]
)

Epoch 1/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 191s 121ms/step - accuracy: 0.7366 - loss: 0.8182 - val_accuracy: 0.8988 - val_loss: 0.3048 - learning_rate: 1.0000e-04
Epoch 2/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 168s 119ms/step - accuracy: 0.8463 - loss: 0.4661 - val_accuracy: 0.9180 - val_loss: 0.2522 - learning_rate: 1.0000e-04
Epoch 3/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 163s 116ms/step - accuracy: 0.8770 - loss: 0.3745 - val_accuracy: 0.9148 - val_loss: 0.2476 - learning_rate: 1.0000e-04
Epoch 4/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 162s 115ms/step - accuracy: 0.8942 - loss: 0.3191 - val_accuracy: 0.9160 - val_loss: 0.2455 - learning_rate: 1.0000e-04
Epoch 5/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 161s 114ms/step - accuracy: 0.9102 - loss: 0.2706 - val_accuracy: 0.9318 - val_loss: 0.2130 - learning_rate: 1.0000e-04
Epoch 6/10
1407/1407 ━━━━━━━━━━━━━━━━━━━━ 159s 113ms/step - accuracy: 0.9242 - loss: 0.2284 - val_accuracy: 0.9278 - val_loss: 0.2093 - learning_rate: 1.0000e-04
Epoch 7/10
1407/1407 ━━━━━━━

Test Accuracy + Summary + Plot

In [ ]:
test_loss, test_accuracy = model.evaluate(
    test_generator,
    verbose=1
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

model.summary()

plot_model(
    model,
    to_file='resnet50_model.png',
    show_shapes=True,
    show_layer_names=True
)